# 📈 Regresión — Prediciendo el NBI a partir de Servicios Urbanos
### DiploDatos 2026 — FAMAF / Universidad Nacional de Córdoba

¿Podemos predecir el nivel de pobreza estructural (NBI) de un barrio a partir de su acceso a servicios públicos?

En este notebook comparamos varios modelos de regresión y analizamos cuáles variables son más importantes.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../data/processed/dataset_final_v6.csv')
print(f'Dataset: {len(df)} barrios, {df.shape[1]} columnas')

## 1. Preparar los datos

In [ ]:
FEATURES = [
    'poblacion',
    'escuelas_total', 'escuelas_estatales', 'escuelas_privadas',
    'centros_salud', 'paradas_colectivo', 'lineas_colectivo',
    'luminarias_reportes', 'comisarias',
]
FEATURES = [f for f in FEATURES if f in df.columns]
TARGET = 'pct_nbi'

# Solo barrios con NBI conocido
df_model = df.dropna(subset=[TARGET]).copy()
df_model[FEATURES] = df_model[FEATURES].fillna(0)

X = df_model[FEATURES].values
y = df_model[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train)} barrios | Test: {len(X_test)} barrios')
print(f'Features: {FEATURES}')
print(f'Target: {TARGET} (media={y.mean():.1f}%, std={y.std():.1f}%)')

## 2. Comparar modelos de regresión

In [ ]:
MODELOS = {
    'Regresión Lineal': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge (L2)':       Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Random Forest':    RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

resultados = []
for nombre, modelo in MODELOS.items():
    cv_scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring='r2')
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    resultados.append({
        'Modelo': nombre,
        'R² (CV train)': cv_scores.mean().round(3),
        'R² (test)': r2_score(y_test, y_pred).round(3),
        'MAE (test)': mean_absolute_error(y_test, y_pred).round(2),
        'RMSE (test)': np.sqrt(mean_squared_error(y_test, y_pred)).round(2),
    })

res_df = pd.DataFrame(resultados).set_index('Modelo')
print('Comparativa de modelos:')
print(res_df.to_string())

## 3. Predicciones vs valores reales (mejor modelo)

In [ ]:
# Usar el mejor modelo (generalmente Random Forest o Gradient Boosting)
mejor_nombre = res_df['R² (test)'].idxmax()
mejor_modelo = MODELOS[mejor_nombre]
mejor_modelo.fit(X_train, y_train)
y_pred_test = mejor_modelo.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicho vs Real
lim_max = max(y_test.max(), y_pred_test.max()) + 2
axes[0].scatter(y_test, y_pred_test, alpha=0.6, color='steelblue', s=40)
axes[0].plot([0, lim_max], [0, lim_max], 'r--', linewidth=1.5, label='Perfecta predicción')
axes[0].set_xlabel('% NBI Real')
axes[0].set_ylabel('% NBI Predicho')
axes[0].set_title(f'{mejor_nombre}\nPredicho vs Real', fontsize=11, fontweight='bold')
axes[0].legend()
r2 = r2_score(y_test, y_pred_test)
axes[0].text(0.05, 0.9, f'R² = {r2:.3f}', transform=axes[0].transAxes, fontsize=11)

# Histograma de residuos
residuos = y_test - y_pred_test
axes[1].hist(residuos, bins=25, color='salmon', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_xlabel('Residuo (Real - Predicho)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Residuos', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../tmp/regresion_prediccion.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Mejor modelo: {mejor_nombre}')
print(f'  R² test  : {r2:.3f}')
print(f'  MAE test : {mean_absolute_error(y_test, y_pred_test):.2f}%')
print(f'  RMSE test: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}%')

## 4. Importancia de variables — ¿Qué predice mejor el NBI?

In [ ]:
# Importancia de features (Random Forest o Gradient Boosting)
modelo_tree = MODELOS.get('Random Forest') or MODELOS.get('Gradient Boosting')
modelo_tree.fit(X, y)
importancias = pd.Series(
    modelo_tree.feature_importances_
    if hasattr(modelo_tree, 'feature_importances_') else
    np.abs(modelo_tree.named_steps['model'].coef_),
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if imp > importancias.median() else '#3498db' for imp in importancias]
ax.barh(importancias.index, importancias.values, color=colors)
ax.set_xlabel('Importancia relativa')
ax.set_title('Importancia de Variables para Predecir el NBI\n(Random Forest)',
             fontsize=12, fontweight='bold')
for i, (idx, val) in enumerate(importancias.items()):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../tmp/regresion_importancia.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Barrios con mayor error de predicción

In [ ]:
# Analizar los barrios donde el modelo se equivoca más
mejor_modelo.fit(X, y)  # re-entrenar con todos los datos
df_model['pct_nbi_pred'] = mejor_modelo.predict(X)
df_model['residuo'] = df_model[TARGET] - df_model['pct_nbi_pred']
df_model['residuo_abs'] = df_model['residuo'].abs()

print('Barrios con mayor error de predicción (outliers):')
outliers = df_model.nlargest(10, 'residuo_abs')[['barrio', 'pct_nbi', 'pct_nbi_pred', 'residuo', 'poblacion']].round(1)
print(outliers.to_string(index=False))
print()
print('📌 Estos barrios tienen características únicas no capturadas por las features actuales.')
print('   Podrían necesitar variables adicionales (salud mental, espacios verdes, etc.)')

## 📝 Conclusiones de la Regresión

1. **Random Forest y Gradient Boosting** son los modelos con mejor performance.
2. La **población del barrio** y las variables de **escuelas y transporte** son los mejores predictores del NBI.
3. El modelo tiene limitaciones: hay barrios con características particulares (zonas industriales, barrios militares, countries) que no siguen el patrón general.
4. **Próximos pasos:** incluir variables de espacios verdes, distancia al centro, densidad habitacional.

---
*Proyecto de Mentoría DiploDatos 2026 — FAMAF/UNC — Mentor: Eber Coronel*